In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier
import time
import psutil
import warnings
warnings.filterwarnings("ignore")

In [2]:
folder_path = 'Processed_Datasets'
file_names = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

dataframes = []
for file_name in file_names:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)
shuffled_df = shuffle(combined_df, random_state=42)

print(shuffled_df.shape)

(4000000, 40)


In [3]:
X = shuffled_df.drop('Label', axis=1)
y = shuffled_df['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [14]:
start_time = time.time()
cpu_usage_start = psutil.cpu_percent(interval=1)
memory_usage_start = psutil.virtual_memory().percent

lgbm = LGBMClassifier(n_estimators=430, learning_rate=0.3, num_leaves=180, max_depth=60, subsample=0.3, colsample_bytree=0.5, reg_alpha=0.5, reg_lambda=0.5, random_state=42)

lgbm.fit(X_train, y_train)

end_time = time.time()
cpu_usage_end = psutil.cpu_percent(interval=1)
memory_usage_end = psutil.virtual_memory().percent

training_time = end_time - start_time
cpu_usage_diff = cpu_usage_end - cpu_usage_start
memory_usage_diff = memory_usage_end - memory_usage_start

print(f"Model training took {training_time:.2f} seconds")
print(f"CPU Usage during training: {cpu_usage_start}% -> {cpu_usage_end}% (Change: +{cpu_usage_diff}%)")
print(f"Memory Usage during training: {memory_usage_start}% -> {memory_usage_end}% (Change: +{memory_usage_diff}%)")

y_pred = lgbm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.8f}')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4156
[LightGBM] [Info] Number of data points in the train set: 2800000, number of used features: 39
[LightGBM] [Info] Start training from score -2.077860
[LightGBM] [Info] Start training from score -2.077857
[LightGBM] [Info] Start training from score -2.079313
[LightGBM] [Info] Start training from score -2.080007
[LightGBM] [Info] Start training from score -2.081372
[LightGBM] [Info] Start training from score -2.079842
[LightGBM] [Info] Start training from score -2.079524
[LightGBM] [Info] Start training from score -2.079762
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [5]:
y_pred = lgbm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.8917


In [6]:
report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.82      0.82    149446
           1       0.99      1.00      0.99    149445
           2       0.86      0.72      0.78    149955
           3       0.76      0.88      0.81    150198
           4       1.00      1.00      1.00    150675
           5       0.83      0.79      0.81    150140
           6       0.93      0.93      0.93    150029
           7       0.97      0.99      0.98    150112

    accuracy                           0.89   1200000
   macro avg       0.89      0.89      0.89   1200000
weighted avg       0.89      0.89      0.89   1200000

